In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import os

# ==========================================
# 1. SETUP E CARICAMENTO MODELLO (4-BIT)
# ==========================================

# NOTA: Per eseguire questo codice è necessaria una GPU con ~4GB-8GB VRAM.
# Utilizziamo Qwen2.5-1.5B-Instruct che è molto leggero (~3GB in 16-bit) e non è 'gated'.
model_id = "Qwen/Qwen2.5-1.5B-Instruct" 

# Rilevamento della GPU e configurazione caricamento
use_4bit = False
torch_dtype = torch.float32 # Default fallback

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0).upper()
    print(f"GPU rilevata: {gpu_name}")
    if "NVIDIA" in gpu_name:
        use_4bit = True
    else:
        # Fallback per AMD/altre GPU: usiamo float16 o bfloat16 per risparmiare memoria
        torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        print(f"GPU non NVIDIA rilevata. Disattivazione bitsandbytes e uso di {torch_dtype}.")
else:
    print("Nessuna GPU CUDA rilevata. Tentativo di caricamento su CPU (richiede molta RAM).")

# Configurazione per la quantizzazione 4-bit (solo per NVIDIA via bitsandbytes)
bnb_config = None
if use_4bit:
    print(f"Attivazione bitsandbytes 4-bit per {model_id}...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

print(f"Caricamento del modello {model_id}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=None if use_4bit else torch_dtype, # bitsandbytes gestisce il dtype internamente
        device_map="auto",
        trust_remote_code=True
    )
except Exception as e:
    print(f"Errore caricamento modello: {e}")
    if use_4bit:
        print("\n--- SUGGERIMENTO: Se l'errore riguarda bitsandbytes, prova a installarlo con: pip install -U bitsandbytes ---")
    model = None

# ==========================================
# 2. RAG PIPELINE (KNOWLEDGE BASE)
# ==========================================

# Documenti di conoscenza per il sistema RAG (RIPRISTINATI)
knowledge_base = [
    "L'IA generativa si basa su modelli probabilistici che predicono il token successivo in una sequenza.",
    "Il Retrieval-Augmented Generation (RAG) riduce le allucinazioni fornendo documenti pertinenti nel prompt.",
    "Llama 3 e Zephyr sono modelli ottimizzati per seguire istruzioni e compiti di reasoning.",
    "La quantizzazione a 4-bit (NF4) permette di ridurre l'uso della VRAM senza perdere troppa precisione.",
    "Il Chain of Thought (CoT) è una tecnica dove si chiede al modello di spiegare i passaggi logici intermedi.",
    "Le emergent properties negli LLM appaiono tipicamente quando il modello supera una certa soglia di parametri."
]

print("Inizializzazione della Vector Search (FAISS)...")
# Forziamo il dispositivo per evitare errori con torch.distributed
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
doc_embeddings = embedder.encode(knowledge_base)

# Creazione dell'indice FAISS per la ricerca di similarità
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings).astype('float32'))

def retrieve_context(query, k=2):
    """Recupera i k documenti più pertinenti alla query."""
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k)
    context = "\n".join([knowledge_base[idx] for idx in indices[0]])
    return context

# ==========================================
# 3. GENERAZIONE CON REASONING (CoT + RAG)
# ==========================================

def generate_response(user_query, system_prompt="Sei un esperto di AI docente di Epicode.", use_rag=True):
    if model is None: return "ERRORE: Modello non caricato. Verifica i requisiti tecnici (GPU + bitsandbytes)."
    
    context = ""
    if use_rag:
        context = retrieve_context(user_query)
    
    # Messaggi per il template (Zephyr usa un formato simile a Llama 3)
    messages = [
        {"role": "system", "content": f"{system_prompt}\nSe usi informazioni dal contesto, cita la fonte.\nSe la domanda è complessa, usa il ragionamento Chain of Thought (pensa passo dopo passo)."},
        {"role": "user", "content": f"CONTESTO:\n{context}\n\nDOMANDA: {user_query}"}
    ]
    
    # Applichiamo il chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generazione con parametri di controllo
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# ==========================================
# 4. SENTIMENT ANALYSIS ZERO-SHOT (OUTPUT JSON)
# ==========================================

def analyze_sentiment_json(text):
    """Esegue sentiment analysis forzando l'output in formato JSON."""
    system_msg = "Sei un analista di dati. Rispondi esclusivamente in formato JSON."
    user_msg = f"Analizza il sentiment di questa recensione: '{text}'. Rispondi solo con un JSON: {{\"sentiment\": \"positivo/negativo\", \"score\": 0.0-1.0}}"
    
    # Prompt manuale per forzare il formato (senza RAG)
    response = generate_response(user_msg, system_prompt=system_msg, use_rag=False)
    
    try:
        # Pulizia della stringa per trovare il JSON
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        json_str = response[json_start:json_end]
        return json.loads(json_str)
    except:
        return {"error": "Impossibile parsare JSON", "raw_response": response}

# ==========================================
# 5. ESECUZIONE TEST
# ==========================================

if __name__ == "__main__":
    if model:
        # Test 1: RAG + Reasoning
        q1 = "Come posso far girare un modello grande su una GPU con poca memoria?"
        print(f"\n[QUERY 1]: {q1}")
        print("-" * 30)
        print(generate_response(q1))
        
        # Test 2: Sentiment Analysis
        review = "Il modulo sugli LLM è stato molto intenso ma estremamente gratificante."
        print(f"\n[REVIEW]: {review}")
        print("-" * 30)
        sentiment = analyze_sentiment_json(review)
        print(f"Sentiment JSON: {sentiment}")
    else:
        print("\n[!] Esecuzione terminata: il modello LLM non è stato caricato correttamente.")
        print("[!] Nota: La parte RAG (Vector Search) invece è stata inizializzata con successo.")


f:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1011 19:31:37.012000 16048 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


GPU rilevata: QUADRO P4000
GPU non NVIDIA rilevata. Disattivazione bitsandbytes e uso di torch.bfloat16.
Caricamento del modello Qwen/Qwen2.5-1.5B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:21<00:00, 16.02it/s]


Inizializzazione della Vector Search (FAISS)...


f:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in F:\EPICODE\hf_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 695.86it/s]



[QUERY 1]: Come posso far girare un modello grande su una GPU con poca memoria?
------------------------------
